In [20]:
import pandas as pd
from pathlib import Path

def _find_header_row(df_raw: pd.DataFrame) -> int:
    for i in range(min(len(df_raw), 300)):
        row = df_raw.iloc[i].astype(str).str.strip().str.lower().tolist()
        has_data = any("data" in c for c in row)
        has_reading_cols = any(
            any(tok in c for tok in ("vazio", "ponta", "cheias", "energia"))
            for c in row
        )
        if has_data and has_reading_cols:
            return i
    raise ValueError("Não encontrei a linha de cabeçalho (procurei por 'data' + 'vazio/ponta/cheias/energia').")

def _pick_data_col(cols):
    for c in cols:
        if isinstance(c, str) and "data" in c.lower():
            return c
    raise KeyError(f"Nenhuma coluna que contenha 'data' foi encontrada nas colunas: {list(cols)}")

def _coerce_reading_cols(df: pd.DataFrame):
    """
    Tenta converter para numérico as colunas de leituras acumuladas.
    1º: pelas palavras-chave; 2º: heurística por conteúdo (>=70% das células parecem números).
    Retorna (df, lista_de_colunas_numericas).
    """
    df = df.copy()
    cand = [c for c in df.columns if any(tok in str(c).lower() for tok in ("vazio","ponta","cheia","cheias","energia","kwh"))]

    # Heurística extra se não encontrarmos nomes óbvios
    if not cand:
        for c in df.columns:
            s = df[c].dropna().astype(str).str.replace("\u00A0","", regex=False).str.strip()
            if len(s) == 0: 
                continue
            looks_num = s.str.fullmatch(r"-?\d{1,3}(\.\d{3})*(,\d+)?|-?\d+([.,]\d+)?").mean()
            if looks_num >= 0.7:
                cand.append(c)

    # Converter candidatos
    for c in cand:
        df[c] = (df[c].astype(str)
                        .str.replace("\u00A0","", regex=False)  # NBSP
                        .str.replace(" ", "", regex=False)
                        .str.replace(".", "", regex=False)      # remover milhar
                        .str.replace(",", ".", regex=False))    # vírgula -> ponto
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Ficar só com os que realmente ficaram numéricos
    num_cols = [c for c in cand if pd.api.types.is_numeric_dtype(df[c])]
    return df, num_cols

def ler_tabela(path_xlsx: str) -> pd.DataFrame:
    df_raw = pd.read_excel(path_xlsx, header=None)
    header_row = _find_header_row(df_raw)
    cols = df_raw.iloc[header_row].tolist()

    df = df_raw.iloc[header_row + 1:].copy()
    df.columns = cols

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)

    data_col = _pick_data_col(df.columns)

    ts = pd.to_datetime(df[data_col].astype(str).str.strip(), dayfirst=True, errors="coerce")
    df = df[~ts.isna()].copy()
    df["__ordem__"] = ts.loc[~ts.isna()]
    df = df.sort_values("__ordem__").drop(columns=["__ordem__"]).reset_index(drop=True)
    return df

def _interpolar_cumulativos(df: pd.DataFrame, data_col: str, force_monotonic: bool = True) -> pd.DataFrame:
    df = df.copy()

    # 1) garantir datetime por dia
    ds = pd.to_datetime(df[data_col].astype(str).str.strip(), dayfirst=True, errors="coerce").dt.normalize()
    df = df.loc[~ds.isna()].copy()
    df[data_col] = ds
    df = df.sort_values(data_col).reset_index(drop=True).set_index(data_col)

    # 2) numerificar leituras
    df, reading_cols = _coerce_reading_cols(df)

    # 3) reindex diário e interpolar por tempo só nas colunas de leitura
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
    df = df.reindex(full_idx)

    if reading_cols:
        df[reading_cols] = df[reading_cols].interpolate(method="time", limit_direction="both", limit_area="inside")

        if force_monotonic:
            for c in reading_cols:
                df[c] = df[c].cummax()  # evita pequenas quebras por ruído de floating

        # opcional: arredondar e voltar a inteiros “Int64” (com NA seguro)
        for c in reading_cols:
            df[c] = df[c].round(0).astype("Int64")

    df = df.reset_index().rename(columns={"index": data_col})
    return df, reading_cols

def juntar_ficheiros(diretorio: str,
                     saida_xlsx: str = "consumos_merged.xlsx",
                     padrao: str = "L*.xlsx",
                     interpolar: bool = True) -> pd.DataFrame:
    base = Path(diretorio)
    files = sorted(base.glob(padrao))
    if not files:
        raise FileNotFoundError(f"Nenhum ficheiro encontrado em {base} com padrão {padrao}")

    dfs = []
    for f in files:
        try:
            df = ler_tabela(str(f))
            dfs.append(df)
            print(f"[OK] {f.name}: {len(df)} registos")
        except Exception as e:
            print(f"[ERRO] {f.name}: {e}")

    if not dfs:
        raise RuntimeError("Nenhum ficheiro foi processado com sucesso.")

    df_all = pd.concat(dfs, ignore_index=True)

    data_col = _pick_data_col(df_all.columns)
    ts_all = pd.to_datetime(df_all[data_col].astype(str).str.strip(), dayfirst=True, errors="coerce")
    df_all = df_all[~ts_all.isna()].copy()
    df_all["__ordem__"] = ts_all.loc[~ts_all.isna()]
    df_all = df_all.sort_values("__ordem__").drop(columns=["__ordem__"]).reset_index(drop=True)

    reading_cols = []
    if interpolar:
        df_all, reading_cols = _interpolar_cumulativos(df_all, data_col)

    # ——— Exportar bem formatado (sem “########”) ———
    out = Path(saida_xlsx)
    with pd.ExcelWriter(out, engine="xlsxwriter", datetime_format="yyyy-mm-dd", date_format="yyyy-mm-dd") as xlw:
        df_all.to_excel(xlw, index=False, sheet_name="merged")
        ws = xlw.sheets["merged"]
        book = xlw.book
        date_fmt = book.add_format({"num_format": "yyyy-mm-dd"})
        int_fmt  = book.add_format({"num_format": "0"})
        # largura confortável para a primeira coluna (data) e restantes
        ws.set_column(0, 0, 12, date_fmt)        # data
        for j, col in enumerate(df_all.columns[1:], start=1):
            ws.set_column(j, j, 14, int_fmt if col in reading_cols else None)

    print(f"\n✅ Ficheiro gerado: {out.resolve()}")
    print(f"   Total de registos (após interpolação): {len(df_all)}")
    return df_all


In [21]:

df = juntar_ficheiros("data", "merged_leituras.xlsx", "L*.xlsx", interpolar=True)

[OK] L0102.xlsx: 59 registos
[OK] L0304.xlsx: 60 registos
[OK] L0506.xlsx: 61 registos
[OK] L07.xlsx: 31 registos
[OK] L08.xlsx: 31 registos

✅ Ficheiro gerado: /Users/miguelferreira/Desktop/SLB/ElectricitySpotPrices/Simulation/nb/merged_leituras.xlsx
   Total de registos (após interpolação): 243


/Users/miguelferreira/miniforge3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/miguelferreira/miniforge3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/miguelferreira/miniforge3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/miguelferreira/miniforge3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/miguelferreira/miniforge3

In [25]:
df[df['Data da Leitura'] >= '2025-04-20' ]

,Data da Leitura,Tipo de Leitura,Origem,Estado,,Vazio,Ponta,Cheias
109,2025-04-20,Real,Operador de Rede de Distribuição,Válida,Energia consumida,5569,2736,6603
110,2025-04-21,NaN,NaN,NaN,NaN,5578,2742,6617
111,2025-04-22,Real,Operador de Rede de Distribuição,Válida,Energia consumida,5588,2747,6631
112,2025-04-23,Real,Operador de Rede de Distribuição,Válida,Energia consumida,5596,2750,6638
113,2025-04-24,Real,Operador de Rede de Distribuição,Válida,Energia consumida,5607,2753,6644
...,...,...,...,...,...,...,...,...
238,2025-08-27,Real,Operador de Rede de Distribuição,Válida,Energia consumida,6208,3050,7332
239,2025-08-28,Real,Operador de Rede de Distribuição,Válida,Energia consumida,6212,3052,7335
240,2025-08-29,Real,Operador de Rede de Distribuição,Válida,Energia consumida,6215,3055,7339
241,2025-08-30,Real,Operador de Rede de Distribuição,Válida,Energia consumida,6219,3056,7342
